# 16. Иерархический Span NER: coarse → fine

Модель сохраняет исходные классы `ACT`, `BIN`, `CMP`, `QUA`, `ECO`, `SOC`, `MET`, `INST`. Внутренние суперклассы используются только для первого этапа и auxiliary loss.

- этап 1: 5 эпох `coarse CE + coarse supervised contrastive loss`;
- этап 2: до 20 эпох `fine CE + coarse auxiliary loss + fine supervised contrastive loss`;
- early stopping второго этапа разрешён только после 15 fine-эпох;
- на Google Drive копируется только один лучший fine-checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/hierarchical_span_ner_corrected_v1.yaml'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'
RUNTIME_OUTPUT = Path('/content/rurebus_runs/hierarchical_span_ner_corrected_v1/seed_42')
DRIVE_OUTPUT = PROJECT_DIR / 'results/hierarchical_span_ner_corrected_v1/seed_42'

for required_path in (EXPERIMENT_CONFIG, BOOTSTRAP):
    if not required_path.is_file():
        raise FileNotFoundError(f'Не найден {required_path}. Обновите проект в Google Drive.')

bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Для иерархического Span NER выберите GPU runtime в Colab.')

In [ ]:
from rurebus_ie.configuration import load_experiment_bundle

bundle = load_experiment_bundle(EXPERIMENT_CONFIG, project_root=PROJECT_DIR)
hierarchy = bundle['model_config']['hierarchy']
print('Внутренние группы:', hierarchy['superclass_groups'])
print('Контрастивный вес:', hierarchy['contrastive_weight'])
print('Fine-классы остаются:', bundle['data_config']['entities']['labels'])

In [ ]:
from rurebus_ie.training import train_hierarchical_span_ner_experiment

summary = train_hierarchical_span_ner_experiment(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    output_dir_override=RUNTIME_OUTPUT,
)
print(f'Лучшая общая эпоха: {summary.best_epoch}')
print(f'Fine validation strict micro-F1: {summary.best_validation_f1:.4f}')
print(f'Локальный checkpoint: {summary.checkpoint_dir}')

In [ ]:
import pandas as pd

history = pd.DataFrame(summary.history)
display(history)
history.plot(x='epoch', y=['train_loss', 'train_coarse_loss', 'train_fine_loss', 'train_contrastive_loss'], grid=True);
history.plot(x='epoch', y=['validation_coarse_micro_f1', 'validation_fine_micro_f1'], ylim=(0, 1), grid=True);

## Однократный экспорт в Google Drive

В runtime-каталоге хранится только лучший fine-checkpoint. Существующий Drive-run намеренно не перезаписывается, чтобы Google Drive не сохранял скрытые версии больших файлов весов.

In [ ]:
import shutil

if DRIVE_OUTPUT.exists():
    raise FileExistsError(
        f'{DRIVE_OUTPUT} уже существует. Задайте новый seed/run или удалите старый осознанно.'
    )
DRIVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUNTIME_OUTPUT, DRIVE_OUTPUT)
print('Итоговый run сохранён один раз:', DRIVE_OUTPUT)